In [16]:
import json
from IPython.display import Markdown
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()
client = Anthropic()

In [17]:
model = "claude-haiku-4-5"
max_tokens = 1024
temperature = 0.45
system_prompt = ""
stop_sequences = ["```"]

In [18]:
# Create Dataset using a prompt

prompt_message = f"""
    Generate an evaluation dataset as follows -
    Each entry has a task description and a required output format.

    Produce exactly 3 tasks. Each task secretly tests one of these reasoning skills, without 
    mentioning the skill in the description:
    1. Factuality / Misconception (TruthfulQA style)
    2. Date / Time Reasoning (Date Understanding style)
    3. Multi-hop Reasoning (HotpotQA style)

    Output exactly one JSON array of 3 objects. Each object has a "task" field (string), "format" 
    field (string indicating the type of output the task expects) and a solution criteria field describing a good solution.

    Requirements:
    - Each task description must be self-contained.
    - Tasks should be realistic and non-trivial.
    - Do not include examples or answers in the task.
    """

In [19]:
messages = []
def append_message(role, message):
    new_message = {"role": role, "content": message}
    messages.append(new_message)
    return messages

In [ ]:
def chat(message: str, temperature: float, system_prompt: str = "", stop_sequences: list = []):
    append_message("user", message)
    prefill = "```json"
    new_message = messages + [{"role": "assistant", "content": prefill}]
    response = client.messages.create(
        model=model, max_tokens=max_tokens, messages=new_message,
        system=system_prompt, temperature=temperature, stop_sequences=stop_sequences
    )
    reply = response.content[0].text
    append_message("assistant", reply)
    return reply

In [21]:
# Create Evaluation Dataset
response = chat(
                message=prompt_message,
                temperature=temperature,
                stop_sequences=stop_sequences
            )

print(response)


[
  {
    "task": "A company's annual report states that their revenue in Q3 2023 was $50 million. However, a financial analyst claims this figure is incorrect based on publicly available data. Explain whether the company's reported figure is accurate or misleading, and provide the reasoning for your conclusion.",
    "format": "A written explanation (2-3 sentences) stating whether the claim is accurate or inaccurate, with clear reasoning about why the statement may or may not reflect reality.",
    "solution_criteria": "The response should demonstrate awareness of common financial reporting misconceptions, distinguish between reported vs. actual figures, and acknowledge potential sources of discrepancy (e.g., accounting methods, currency differences, or data sources). A good solution avoids accepting claims at face value and considers multiple perspectives on factuality."
  },
  {
    "task": "An event is scheduled to occur 45 days after March 15th, 2024. A participant claims they wi

In [22]:
with open("dataset.json", "w") as f:
    json_res = json.loads(response)
    json.dump(json_res, f, indent=2)

In [23]:
# Execute each task in the dataset
def run_prompt(test_case):
    messages.clear()  # isolate each task from dataset generation context and prior tasks
    prompt = f"""
        Please solve the following task:
        {test_case["task"]}
        * Respond in correct format as applicable to the question
        * Do not add any comments or explanation
        """
    output = chat(prompt, temperature=temperature, stop_sequences=stop_sequences)
    return output

In [24]:
# Model grader

def grade_by_model(test_case, output):
    evaluation_prompt = f"""
                            You are an expert evaluator. Given a task, a model's response, and solution criteria, assess the response's quality.
                            Task: {test_case["task"]}
                            Expected output format: {test_case["format"]}
                            Model's response: {output}
                            Solution criteria (the response should meet these): {test_case["solution_criteria"]}
                            Evaluate the response. Provide:
                            - A score from 1 to 10 (1 = fails to meet criteria, 10 = fully meets all criteria)
                            - A concise justification explaining strengths and weaknesses relative to each criterion point
                            - An overall verdict (PASS if score >= 4, else FAIL)
                            Output as JSON:
                            {{
                            "score": int,
                            "justification": "string",
                            "verdict": "PASS or FAIL"
                            }}
                        """
    grader_messages = [{"role": "user", "content": evaluation_prompt},
                       {"role": "assistant", "content": "```json"}]
    response = client.messages.create(
        model=model, max_tokens=1024, messages=grader_messages,
        temperature=0, stop_sequences=["```"]
    )
    return json.loads(response.content[0].text)

In [27]:
# Load Test Cases
dataset = None
with open("dataset.json", "r") as f:
    dataset = json.load(f)

# Execute each case
outputs = []
for test_case in dataset:
    model_output = run_prompt(test_case)
    outputs.append(model_output)

# print(json.dumps(outputs, indent=2))


In [26]:
# Grade By Model
grades = []
for i in range(len(dataset)):
    grade = grade_by_model(dataset[i], outputs[i])
    grades.append(grade)
    print(f"Task {i+1}: score={grade['score']}, verdict={grade['verdict']}")
    print(f"  {grade['justification']}\n")

Task 1: score=6, verdict=PASS
  The model's response demonstrates several strengths and weaknesses relative to the solution criteria:

Strengths:
1. Appropriately acknowledges insufficient information rather than making unfounded claims, showing critical thinking
2. Correctly identifies that verification requires specific data sources and evidence
3. Recognizes the need for the company name and analyst's alternative figures for proper assessment
4. Avoids accepting claims at face value, which aligns with the criterion about not accepting claims uncritically

Weaknesses:
1. Does not demonstrate awareness of common financial reporting misconceptions (e.g., GAAP vs. non-GAAP reporting, revenue recognition timing, currency conversions, consolidation methods, one-time items)
2. Fails to distinguish between reported vs. actual figures in a substantive way - merely states data is needed rather than explaining what types of discrepancies commonly exist
3. Does not acknowledge potential sources